In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l0.1 Generate text

Train PocketLM on Shakespeare and watch it speak. It is tiny and barely
trained, so expect gibberish with the *shape* of English: line breaks,
capitalized speaker names, spacing. That shape is the model learning.

In [ ]:
from pocketlm import train_tiny, pick_config, generate

corpus = open(CORPUS_PATH, encoding="utf-8").read()
model, tok = train_tiny(corpus, pick_config(DEVICE, 1), seed=0)
out = generate(model, tok, "ROMEO:", max_new_tokens=200, seed=0)
print(out)

Read the stream above. Even a 2-layer model trained for a handful of
steps reproduces the *texture* of the corpus. The next lessons open up
why: tokens, probabilities, and how decoding turns probabilities into text.

In [ ]:
# CI guard: generation never changes length or emits non-vocab characters.
assert len(out) == len("ROMEO:") + 200
assert set(out) <= set(tok.itos)